# 🛡️ Lexsi Agent Circuit Defense: Colab Execution Notebook
### Provable Indirect Prompt Injection Defense via Subspace Weight Surgery
**Hackathon:** Lexsi AI Safety & Alignment Hackathon (Track 2: AI Red Teaming, Safety & Alignment)
**Toolchain:**  →  →  → 

> **GPU Recommendation:**
> - **Colab Pro (A100 / L4 / V100):** Run full .
> - **Colab Free (T4 16GB):** Run  (fits in 15GB VRAM bfloat16) or smoke test with .


### 1. Verify GPU Hardware


In [ ]:
!nvidia-smi


### 2. Clone Repository & Install Dependencies


In [ ]:
# Clone repository and enter directory
!git clone https://github.com/GiGiKoneti/lexsi-agent-circuit-defense.git
%cd lexsi-agent-circuit-defense

# Install PyTorch & required packages
!pip install -q transformers>=4.45.0 accelerate peft safetensors sentencepiece datasets plotly gradio pyyaml tqdm click

# Install Lexsi-Labs toolkits
!pip install -q git+https://github.com/Lexsi-Labs/CuratorKIT.git || true
!pip install -q git+https://github.com/Lexsi-Labs/CircuitKIT.git || true
!pip install -q git+https://github.com/Lexsi-Labs/SafeTune.git || true
!pip install -q git+https://github.com/Lexsi-Labs/AuditKIT.git || true


### 3. Hugging Face Authentication
Enter your Hugging Face Access Token to download Cohere model weights.


In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face Token: ")
login(token=hf_token)


### 4. Choose Model Configuration


In [ ]:
# Model Selection:
# Use "configs/model_aya_8b.yaml" for full 8B model (Requires A100/L4 or T4)
# Use "configs/model_tiny_aya.yaml" for fast smoke testing
CONFIG_PATH = "configs/model_aya_8b.yaml"
print(f"Using configuration: {CONFIG_PATH}")


### 5. Step 1: CuratorKIT Triplet Synthesis & Preprocessing


In [ ]:
!python scripts/01_generate_triplets.py
!python scripts/preprocess_and_validate.py


### 6. Step 2: CircuitKIT EAP-IG Circuit Discovery & Subspace Extraction


In [ ]:
!python scripts/02_run_circuit_discovery.py \n    --config  \n    --triplets_path data/processed/triplets_cleaned.jsonl \n    --output_basis data/processed/basis_matrices_k4.pt \n    --num_samples 100


### 7. Step 3: SafeTune Orthogonal Weight Surgery (^\perp$)


In [ ]:
!python scripts/03_apply_weight_surgery.py \n    --config  \n    --basis_path data/processed/basis_matrices_k4.pt \n    --output_dir safetune_aya8b_projected


### 8. Step 4: AuditKIT Comprehensive Evaluation


In [ ]:
!python scripts/04_run_full_audit.py \n    --base_model CohereLabs/aya-expanse-8b \n    --edited_model ./safetune_aya8b_projected \n    --triplets_path data/processed/triplets_cleaned.jsonl \n    --output_report data/processed/audit_report.json \n    --num_eval_samples 100


### 9. View AuditKIT Benchmark Results


In [ ]:
import json
import pandas as pd
from IPython.display import display

with open("data/processed/audit_report.json") as f:
    report = json.load(f)

df = pd.DataFrame(report["Evaluation_Summary"])
display(df)
